# Sampla — RAVE training for the Sample Librarian neural sampler

Train (or fine-tune) a **RAVE** model on your own one-shots and export the
`rave_encoder.onnx` / `rave_decoder.onnx` pair that the Sample Librarian VST
loads. Runs entirely in **Google Colab on a free GPU** — no local toolchain.

### Before you start
- **Runtime → Change runtime type → GPU** (T4 is fine).
- Best results come from **one coherent category** (all kicks, all vox chops,
  all pads). A few hundred one-shots is plenty to **fine-tune**; training from
  scratch wants more (30 min–hours of audio).
- Training takes a while (often **a few hours** on a free T4). Save checkpoints
  to Google Drive (cell below) so a disconnect doesn't lose progress.

You do **not** need this repo checked out — every step is self-contained here.


## 1. Check the GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime -> Change runtime type -> GPU'
import torch; print('CUDA available:', torch.cuda.is_available())


## 2. Install RAVE

`acids-rave` is the official implementation (ACIDS-IRCAM). This pulls in a
compatible PyTorch + ONNX toolchain.


In [ ]:
!pip -q install acids-rave onnx
!rave --help >/dev/null 2>&1 && echo 'rave CLI ready' || echo 'check install output above'


## 3. (Optional) Mount Google Drive for checkpoints

Recommended: training state is written under `/content/drive/MyDrive/rave_runs`,
so you can resume after a disconnect by re-running the train cell.


In [ ]:
USE_DRIVE = True  #@param {type:'boolean'}
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS = '/content/drive/MyDrive/rave_runs'
else:
    RUNS = '/content/rave_runs'
os.makedirs(RUNS, exist_ok=True)
print('runs dir:', RUNS)


## 4. Upload your one-shots

Upload a **.zip** of your samples (one coherent category). WAV/AIFF/FLAC/MP3
are all accepted by the preprocessor. They'll be unpacked to `/content/audio`.


In [ ]:
import os, zipfile, glob
from google.colab import files
os.makedirs('/content/audio', exist_ok=True)
print('Choose your .zip of samples...')
up = files.upload()
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('/content/audio')
        print('unpacked', name)
n = sum(len(glob.glob(f'/content/audio/**/*.{e}', recursive=True))
        for e in ('wav','WAV','aif','aiff','flac','mp3'))
print('audio files found:', n)


## 5. Preprocess into a RAVE dataset

RAVE slices the audio into training windows. Set the sample rate you want the
model to run at (**44100** matches the plugin default; 48000 is also common).


In [ ]:
SR = 44100  #@param {type:'integer'}
!rave preprocess --input_path /content/audio --output_path /content/dataset \
    --channels 1 --sample_rate $SR
print('dataset ready')


## 6. Train / fine-tune

For a few hundred one-shots, **fine-tuning** from a pretrained checkpoint gives
far better results than training from scratch. If you have a pretrained RAVE
`.ckpt` (same sample rate/config), put its path in `CKPT`; leave it blank to
train from scratch.

`MAX_STEPS`: ~200k–500k is reasonable for fine-tuning; scratch training often
needs 1M+. Watch the audio previews in the logs. Re-run this cell to resume
(it picks up from the latest checkpoint under `RUNS`).

> Config note: `v2` is the standard architecture. Some `acids-rave` versions
> name flags slightly differently — run `!rave train --help` if a flag is
> rejected, and adjust.


In [ ]:
NAME = 'my_oneshots'  #@param {type:'string'}
CONFIG = 'v2'         #@param {type:'string'}
MAX_STEPS = 300000    #@param {type:'integer'}
CKPT = ''             #@param {type:'string'}
ckpt_arg = f'--ckpt {CKPT}' if CKPT else ''
!rave train --config $CONFIG --db_path /content/dataset --name $NAME \
    --out_path $RUNS --channels 1 --max_steps $MAX_STEPS $ckpt_arg
print('training stopped (finished or interrupted)')


## 7. Export the trained model to TorchScript

`--streaming false` bakes a non-cached graph, which is what converts cleanly
to ONNX. This writes `<NAME>.ts`.


In [ ]:
import glob
run_dir = sorted(glob.glob(f'{RUNS}/{NAME}*'))[-1]
print('exporting run:', run_dir)
!rave export --run "$run_dir" --streaming false
ts = sorted(glob.glob(f'{run_dir}/**/*.ts', recursive=True))
assert ts, 'no .ts produced — check the export output above'
TS_PATH = ts[-1]
print('scripted model:', TS_PATH)


## 8. Convert TorchScript → ONNX (the plugin's format)

Self-contained version of `tools/export_rave_onnx.py`: it scripts thin
encode/decode wrappers (so ONNX export uses RAVE's own graph rather than
tracing) and forces the legacy exporter.


In [ ]:
import torch, os
OUT = '/content/rave_onnx'; os.makedirs(OUT, exist_ok=True)
model = torch.jit.load(TS_PATH, map_location='cpu').eval()

sr = SR
for a in ('sr','sampling_rate','sample_rate'):
    if hasattr(model, a):
        try: sr = int(getattr(model, a)); break
        except Exception: pass

class Encoder(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, audio):
        z = self.m.encode(audio)
        return z[0] if isinstance(z,(tuple,list)) else z
class Decoder(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, latent):
        y = self.m.decode(latent)
        return y[0] if isinstance(y,(tuple,list)) else y

def scripted(w):
    try: return torch.jit.script(w)
    except Exception as e: print('script fallback:', e); return w
enc = scripted(Encoder(model).eval())
dec = scripted(Decoder(model).eval())

audio = torch.zeros(1,1,131072)
with torch.no_grad(): z = enc(audio)
print('latent shape:', tuple(z.shape), 'hop ~=', 131072//max(1,z.shape[-1]))

def export(mod, ex, path, i, o):
    kw = dict(opset_version=17, input_names=[i], output_names=[o],
              dynamic_axes={i:{2:'L'}, o:{2:'L2'}})
    try: torch.onnx.export(mod, ex, path, dynamo=False, **kw)
    except TypeError: torch.onnx.export(mod, ex, path, **kw)
export(enc, audio, f'{OUT}/rave_encoder.onnx', 'audio', 'latent')
export(dec, z,     f'{OUT}/rave_decoder.onnx', 'latent', 'audio')
open(f'{OUT}/rave_sr.txt','w').write(f'{sr}\n')
print('wrote ONNX pair @', sr, 'Hz to', OUT)


## 9. Download the model

Grab the zip, then unzip it **next to the Sample Librarian DLL** together with
`onnxruntime.dll` (the one from the starter pack). The plugin's NN status will
read `NN: ready`.


In [ ]:
import shutil
shutil.make_archive('/content/rave_model','zip',OUT)
from google.colab import files
files.download('/content/rave_model.zip')


## Troubleshooting

- **ONNX export errors** (custom ops / PQMF): make sure step 7 used
  `--streaming false`. If a specific op still won't convert, try `opset 18`,
  or export a `v1`/`onnx`-friendly config. As a fallback, the repo's
  dependency-free small-autoencoder trainer also emits the same ONNX contract.
- **`rave` flag rejected**: versions differ — run `!rave train --help` /
  `!rave export --help` and adjust the flag names.
- **Sample-rate mismatch**: `rave_sr.txt` carries the model rate; the plugin
  resamples your library to it automatically. Nothing to change.
- **Out of memory**: lower the batch size (`--batch N` on train) or the
  dataset window in preprocess.
- **Colab disconnects**: keep `USE_DRIVE = True`; re-run the train cell to
  resume from the last checkpoint.
